In [10]:
import json

In [11]:
with open("reaction.json", "r") as f:
    json_data = json.load(f)

In [ ]:
# Create a lookup index for speed
index = {item.get('entityUrn'): item for item in json_data['included'] if 'entityUrn' in item}

# Extract activities from the main feed list
activities = json_data['data']['data']['feedDashProfileUpdatesByMemberReactions']['*elements']

for urn in activities:
    item = index.get(urn)
    if item and item.get('$type') == 'com.linkedin.voyager.dash.feed.Update':
        action = item['header']['text']['text']
        post_url = item['socialContent']['shareUrl']
        print(f"Action: {action} | Link: {post_url}")

Action: Rob Hochstein likes this | Link: https://www.linkedin.com/posts/alexhormozi_you-can-beat-99-of-people-by-preparing-activity-7383535744910495744-IBg5?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU
Action: Rob Hochstein liked David Falato’s comment on this | Link: https://www.linkedin.com/posts/robhochstein_i-was-recently-interviewed-on-the-garlic-activity-7110649546296229888-DCYF?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU
Action: Rob Hochstein liked Lauren Apple 🍏’s comment on this | Link: https://www.linkedin.com/posts/robhochstein_i-was-recently-interviewed-on-the-garlic-activity-7110649546296229888-DCYF?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU
Action: Rob Hochstein likes this | Link: https://www.linkedin.com/posts/marcuskeithhayes_carmichaelcollege-vandyhre-commercialpainters-ugcPost-73727669407104286

In [20]:
# 1. Create lookup index for speed
index = {item.get('entityUrn'): item for item in json_data['included'] if 'entityUrn' in item}

# 2. Extract activities
activities = json_data.get('data', {}).get('data', {}).get('feedDashProfileUpdatesByMemberReactions', {}).get('*elements', [])

for urn in activities:
    item = index.get(urn)
    if not item or item.get('$type') != 'com.linkedin.voyager.dash.feed.Update':
        continue

    # --- 1. THE ACTION & REACTION TYPE ---
    header_text = item.get('header', {}).get('text', {}).get('text', "")
    reaction_type = "LIKE" # Default
    if "commented" in header_text.lower(): reaction_type = "COMMENT"
    elif "loves" in header_text.lower(): reaction_type = "LOVE"
    elif "insightful" in header_text.lower(): reaction_type = "INSIGHTFUL"

    # --- 2. THE ORIGINAL POST TEXT ---
    # In a reaction update, the original post text is often in 'commentary'
    # or inside 'resharedUpdate' if they reacted to a share.
    post_text = "No text content"
    if item.get('commentary'):
        post_text = item['commentary'].get('text', {}).get('text', "")
    
    # --- 3. THE ORIGINAL AUTHOR ---
    # We look for the 'actor' URN and find their name in our index
    author_name = "Unknown Author"
    actor = item.get('actor')
    author_name = actor["name"]["text"]
    author_headline = actor["description"]["text"]

    # --- 4. THE URL ---
    post_url = item.get('socialContent', {}).get('shareUrl') or item.get('metadata', {}).get('shareUrl')
    
    print(f"--- ACTIVITY FOUND ---")
    print(f"User Action:  {header_text}")
    print(f"Reaction:     {reaction_type}")
    print(f"Post Author:  {author_name}")
    print(f"Author Headline: {author_headline}")
    print(f"Post Content: {post_text[:100]}...") # Showing first 100 chars
    print(f"Link:         {post_url}")
    print("-" * 40)

--- ACTIVITY FOUND ---
User Action:  Rob Hochstein likes this
Reaction:     LIKE
Post Author:  Alex Hormozi
Author Headline: Founder Acquisition.com, Co-Founder Skool.com. Get your free scaling roadmap👇
Post Content: You can beat 99% of people by:

preparing the night before…
showing up early…
leaving only when the ...
Link:         https://www.linkedin.com/posts/alexhormozi_you-can-beat-99-of-people-by-preparing-activity-7383535744910495744-IBg5?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU
----------------------------------------
--- ACTIVITY FOUND ---
User Action:  Rob Hochstein liked David Falato’s comment on this
Reaction:     LIKE
Post Author:  Rob Hochstein
Author Headline: Get More of Your Ideal Customers for Fast, Sustainable Revenue Growth
Post Content: I was recently interviewed on the Garlic Marketing Show Podcast by Ian Garlic! 

We discussed How Li...
Link:         https://www.linkedin.com/posts/robhochstein_i-was-r

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.text import Text
from rich import box

# 1. Create lookup index for speed
index = {item.get('entityUrn'): item for item in json_data['included'] if 'entityUrn' in item}

# 2. Extract activities
activities = json_data.get('data', {}).get('data', {}).get('feedDashProfileUpdatesByMemberReactions', {}).get('*elements', [])

# Initialize rich console
console = Console()

for urn in activities:
    item = index.get(urn)
    if not item or item.get('$type') != 'com.linkedin.voyager.dash.feed.Update':
        continue

    # --- 1. THE ACTION & REACTION TYPE ---
    header_text = item.get('header', {}).get('text', {}).get('text', "")

    # --- 2. THE ORIGINAL POST TEXT ---
    # In a reaction update, the original post text is often in 'commentary'
    # or inside 'resharedUpdate' if they reacted to a share.
    post_text = "No text content"
    if item.get('commentary'):
        post_text = item['commentary'].get('text', {}).get('text', "")
    
    # --- 3. THE ORIGINAL AUTHOR ---
    # We look for the 'actor' URN and find their name in our index
    author_name = "Unknown Author"
    actor = item.get('actor')
    author_name = actor["name"]["text"]
    author_headline = actor["description"]["text"]

    # --- 4. THE URL ---
    post_url = item.get('socialContent', {}).get('shareUrl') or item.get('metadata', {}).get('shareUrl')

    highlightedComments = item.get('*highlightedComments',[])
    if highlightedComments:
        comment_obj = index.get(highlightedComments[0])
        print(highlightedComments[0])
        comment_text = comment_obj.get('commentary', {}).get('text', {}).get('text', "No text")
            # Extract Author
        comment_author = comment_obj.get('commenter', {}).get('title', {}).get('text', "Unknown")
        print(comment_obj)
    
    content = Text()
    content.append("User Action:  ", style="bold cyan")
    content.append(f"{header_text}\n", style="white")
    content.append("Post Author:  ", style="bold cyan")
    content.append(f"{author_name}\n", style="bold white")
    content.append("Post Author Headline: ", style="bold cyan")
    content.append(f"{author_headline}\n", style="dim white")
    content.append("Post Content: ", style="bold cyan")
    content.append(f"{post_text}", style="white")
    content.append("\nLink:         ", style="bold cyan")
    content.append(f"{post_url}", style="blue underline")
    
    panel = Panel(
        content,
        title="[bold magenta]ACTIVITY FOUND[/bold magenta]",
        border_style="magenta",
        box=box.ROUNDED
    )
    
    console.print(panel)
    console.print()  # Empty line for spacing

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Alex Hormozi                                                                                      │
│ Post Author Headline: Founder Acquisition.com, Co-Founder Skool.com. Get your free scaling roadmap👇            │
│ Post Content: You can beat 99% of people by:                                                                    │
│                                                                                                                 │
│ preparing the night before…                                                                                     │
│ showing up early…                                                                                               │
│ leaving only when the job is done exceptionally right…                                                          │
│ remembering people’s names…                                                                                     │
│ following up…                                                                                                   │
│ improving one thing each time...                                                                                │
│                                                                                                                 │
│ Do all of this consistently, every day, for one year.                                                           │
│ I promise you will get closer to your goals.                                                                    │
│                                                                                                                 │
│ - Alex ✊🏽                                                                                                       │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/alexhormozi_you-can-beat-99-of-people-by-preparing-activity-7383535744910495744- │
│ IBg5?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

urn:li:fsd_comment:(7270579058130714626,urn:li:activity:7110649546296229888)
{'connectAction': None, 'annotationActionType': None, 'commentPrompt': None, 'content': None, 'followAction': {'unmuteTrackingActionType': None, 'unfollowTrackingActionType': 'unfollowMember', 'muteTrackingActionType': None, 'followTrackingActionType': 'followMember', 'companyFollowingTrackingContext': None, 'type': 'FOLLOW_TOGGLE', '*followingState': 'urn:li:fsd_followingState:urn:li:fsd_profile:ACoAAAA707sBIVusi0jlYmUeTnlMT8S7jVzEWtM', '$recipeTypes': ['com.linkedin.e1540f67700591f7e1144514598dec1b'], '$type': 'com.linkedin.voyager.dash.feed.actions.follow.FollowAction'}, 'createdAt': 1733441128286, 'entityUrn': 'urn:li:fsd_comment:(7270579058130714626,urn:li:activity:7110649546296229888)', 'parentComment': None, 'headline': None, 'contributed': False, 'trackingId': None, 'annotation': None, 'edited': False, 'threadUrn': None, 'timeOffset': -1, '$recipeTypes': ['com.linkedin.b90d03c0fab8d9eede3ff18a27af7ebe'

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein liked David Falato’s comment on this                                                │
│ Post Author:  Rob Hochstein                                                                                     │
│ Post Author Headline: Get More of Your Ideal Customers for Fast, Sustainable Revenue Growth                     │
│ Post Content: I was recently interviewed on the Garlic Marketing Show Podcast by Ian Garlic!                    │
│                                                                                                                 │
│ We discussed How LinkedIn works for B2B Outbound Sales & Cracking the Code to Waking up to Booked Sales Calls   │
│ Everyday!                                                                                                       │
│                                                                                                                 │
│ Thanks Ian for having me on and hopefully this answers some questions a lot of people ask me in regards to what │
│ actually works and some common issues that can hurt or limit your results.                                      │
│                                                                                                                 │
│ Enjoy and let me know if I can clarify anything we discussed.                                                   │
│                                                                                                                 │
│ Link to full YouTube Interview below in the comments...and I definitely  recommend subscribing to Ian's YouTube │
│ channel to hear from more successful entrepreneurs about how they are growing their businesses.                 │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/robhochstein_i-was-recently-interviewed-on-the-garlic-activity-71106495462962298 │
│ 88-DCYF?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

urn:li:fsd_comment:(7316475933408874497,urn:li:activity:7110649546296229888)
{'connectAction': None, 'annotationActionType': None, 'commentPrompt': None, 'content': None, 'followAction': {'unmuteTrackingActionType': None, 'unfollowTrackingActionType': 'unfollowMember', 'muteTrackingActionType': None, 'followTrackingActionType': 'followMember', 'companyFollowingTrackingContext': None, 'type': 'FOLLOW_TOGGLE', '*followingState': 'urn:li:fsd_followingState:urn:li:fsd_profile:ACoAACOcn_kBgp1JpuoC0PN9o9L_cmKHBWF6ffg', '$recipeTypes': ['com.linkedin.e1540f67700591f7e1144514598dec1b'], '$type': 'com.linkedin.voyager.dash.feed.actions.follow.FollowAction'}, 'createdAt': 1744383796075, 'entityUrn': 'urn:li:fsd_comment:(7316475933408874497,urn:li:activity:7110649546296229888)', 'parentComment': None, 'headline': None, 'contributed': False, 'trackingId': None, 'annotation': None, 'edited': False, 'threadUrn': None, 'timeOffset': -1, '$recipeTypes': ['com.linkedin.b90d03c0fab8d9eede3ff18a27af7ebe'

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein liked Lauren Apple 🍏’s comment on this                                             │
│ Post Author:  Rob Hochstein                                                                                     │
│ Post Author Headline: Get More of Your Ideal Customers for Fast, Sustainable Revenue Growth                     │
│ Post Content: I was recently interviewed on the Garlic Marketing Show Podcast by Ian Garlic!                    │
│                                                                                                                 │
│ We discussed How LinkedIn works for B2B Outbound Sales & Cracking the Code to Waking up to Booked Sales Calls   │
│ Everyday!                                                                                                       │
│                                                                                                                 │
│ Thanks Ian for having me on and hopefully this answers some questions a lot of people ask me in regards to what │
│ actually works and some common issues that can hurt or limit your results.                                      │
│                                                                                                                 │
│ Enjoy and let me know if I can clarify anything we discussed.                                                   │
│                                                                                                                 │
│ Link to full YouTube Interview below in the comments...and I definitely  recommend subscribing to Ian's YouTube │
│ channel to hear from more successful entrepreneurs about how they are growing their businesses.                 │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/robhochstein_i-was-recently-interviewed-on-the-garlic-activity-71106495462962298 │
│ 88-DCYF?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Lindsay Christensen                                                                               │
│ Post Author Headline: Premier IV Hydration Therapy | Anti-Aging | Weight loss | Hangover Recovery | Athletic    │
│ Performance | Immunity Armour | Fatigue Fighter | Jet Lag Recovery                                              │
│ Post Content: I’m happy to share that I’m opening my first business: Prime IV Hydration & Wellness (Geneva,     │
│ IL). Super grateful to Matt for always having my back and giving me the flexibility to find my next adventure❤️  │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/linz-christensen_im-happy-to-share-that-im-opening-my-first-ugcPost-723546391258 │
│ 0595712-YdBD?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfq │
│ NIfU                                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Justin Welsh                                                                                      │
│ Post Author Headline: The $10M Solopreneur | Helping 100,000+ experts turn their expertise into income.         │
│ Post Content: There are 8 simple daily activities that changed my life.                                         │
│                                                                                                                 │
│ (and may change yours)                                                                                          │
│                                                                                                                 │
│ 1. A morning workout                                                                                            │
│ 2. A big, healthy lunch                                                                                         │
│ 3. A small, healthy dinner                                                                                      │
│ 4. 3 hours in the 'genius' zone                                                                                 │
│ 5. 30 minutes of reading                                                                                        │
│ 6. Time with my wife                                                                                            │
│ 7. Time outside                                                                                                 │
│ 8. Time alone                                                                                                   │
│                                                                                                                 │
│ My best days as an entrepreneur have all 8.                                                                     │
│                                                                                                                 │
│ I call this 'life-work integration'.                                                                            │
│                                                                                                                 │
│ Here's how you build it: https://lnkd.in/ea7MU6-8                                                               │
│                                                                                                                 │
│ What do your best days look like?                                                                               │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/justinwelsh_there-are-8-simple-daily-activities-that-activity-722872917276273459 │
│ 2-lyDm?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Matt Orlins                                                                                       │
│ Post Author Headline: Government Relations | Cross-Functional Leadership | Storytelling | Attorney              │
│ Post Content: This week, the U.S. Senate approved versions of the Kids Online Safety Act and the Children’s     │
│ Online Privacy Protection Act, which would represent some of the most significant legislation in decades aimed  │
│ at addressing how platforms can interact with children. Passage in the House remains at best an open question.  │
│ Some members are already expressing serious reservations.                                                       │
│                                                                                                                 │
│ As someone who works in early childhood education policy and as a parent, I am interested to see what approach, │
│ if any, states and the federal government coalesce around to ensure that kids do not experience so many adverse │
│ consequences from spending time online. A couple of weeks ago, Kernard D. Jones and I hosted a roundtable on    │
│ screen usage and mental health focused on early learners at the Education Commission of the States National     │
│ Forum on Education Policy. One takeaway for us is that there’s a ton of interest and energy to do something,    │
│ but that we’re still looking for consensus on what that is.                                                     │
│                                                                                                                 │
│ This week’s legislative wrangling is a reminder, though, that as tech platforms and parents, we don’t have to   │
│ wait. There are things that we can and should do to try to help ensure that any experiences kids do have online │
│ are positive and age appropriate. One of the reasons that I’m proud to work at Waterford.org is that we center  │
│ children in the design of our programs.  Our programs are time limited, highly interactive, age appropriate,    │
│ rooted in the science of learning, and offered free of advertising. Technology can continue to be an essential  │
│ tool for helping children learn, but we need to be proactive to ensure that it is.                              │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/mattorlins_best-practices-for-screen-time-in-early-childhood-ugcPost-72251309146 │
│ 26842625-1Kdt?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWf │
│ qNIfU                                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Justin Welsh                                                                                      │
│ Post Author Headline: The $10M Solopreneur | Helping 100,000+ experts turn their expertise into income.         │
│ Post Content: Stop waiting for your "big break."                                                                │
│                                                                                                                 │
│ Go create it yourself.                                                                                          │
│                                                                                                                 │
│ Most people I know:                                                                                             │
│                                                                                                                 │
│ • Chase trends                                                                                                  │
│ • Wait for opportunities                                                                                        │
│ • Hope for luck                                                                                                 │
│                                                                                                                 │
│ The result?                                                                                                     │
│                                                                                                                 │
│ A future decided by others.                                                                                     │
│                                                                                                                 │
│ Here's a better plan:                                                                                           │
│                                                                                                                 │
│ • Set the trends                                                                                                │
│ • Create opportunities                                                                                          │
│ • Make your own damn luck                                                                                       │
│                                                                                                                 │
│ Want to be successful in 5 years?                                                                               │
│ Start building that success today.                                                                              │
│                                                                                                                 │
│ It's not about predicting the market.                                                                           │
│ It's about *creating* the market.                                                                               │
│                                                                                                                 │
│ And easiest place is right here on LinkedIn.                                                                    │
│                                                                                                                 │
│ Here's how 25,000+ people have done the same: https://lnkd.in/egPyfFmy                                          │
│                                                                                                                 │
│ What future will you start creating today?            

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Leila Hormozi                                                                                     │
│ Post Author Headline: Founder and CEO of Acquisition.com                                                        │
│ Post Content: Just like a fish grows to the limit of its tank.                                                  │
│ A business grows to the limit of one thing.                                                                     │
│                                                                                                                 │
│ The capability of the person leading the company.                                                               │
│ A business grows to the limit of its leader's capability.                                                       │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/leilahormozi_just-like-a-fish-grows-to-the-limit-of-its-activity-722735037931364 │
│ 3521-At34?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIf │
│ U                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
from rich.console import Console
from rich.panel import Panel
from rich.text import Text
from rich import box

# 1. Create lookup index for speed
index = {item.get('entityUrn'): item for item in json_data['included'] if 'entityUrn' in item}

# 2. Extract activities
activities = json_data.get('data', {}).get('data', {}).get('feedDashProfileUpdatesByMemberReactions', {}).get('*elements', [])

# Initialize rich console
console = Console()

for urn in activities:
    item = index.get(urn)
    if not item or item.get('$type') != 'com.linkedin.voyager.dash.feed.Update':
        continue

    # --- 1. THE ACTION & REACTION TYPE ---
    header_text = item.get('header', {}).get('text', {}).get('text', "")

    # --- 2. THE ORIGINAL POST TEXT ---
    # In a reaction update, the original post text is often in 'commentary'
    # or inside 'resharedUpdate' if they reacted to a share.
    post_text = "No text content"
    if item.get('commentary'):
        post_text = item['commentary'].get('text', {}).get('text', "")
    
    # --- 3. THE ORIGINAL AUTHOR ---
    # We look for the 'actor' URN and find their name in our index
    author_name = "Unknown Author"
    author_headline = ""
    actor = item.get('actor')
    if actor:
        author_name = actor.get("name", {}).get("text", "Unknown Author")
        author_headline = actor.get("description", {}).get("text", "")

    # --- 4. THE URL ---
    post_url = item.get('socialContent', {}).get('shareUrl') or item.get('metadata', {}).get('shareUrl')

    # --- 5. EXTRACT COMMENTS ---
    highlightedComments = item.get('*highlightedComments', [])
    comment_panels = []
    
    if highlightedComments:
        for comment_urn in highlightedComments:
            comment_obj = index.get(comment_urn)
            if not comment_obj:
                continue
                
            # Extract comment data
            print(comment_urn)
            comment_text = comment_obj.get('commentary', {}).get('text', {})
            commenter = comment_obj.get('commenter', {})
            comment_author = commenter.get('title', {}).get('text', "Unknown")
            comment_profile_url = commenter.get('navigationUrl', "")
            comment_headline = commenter.get('subtitle', "")
            
            # Create comment content with author info
            comment_content = Text()
            comment_content.append("Comment Author: ", style="bold yellow")
            if comment_profile_url:
                # Rich doesn't support clickable links in Text, so we'll show the URL
                comment_content.append(f"{comment_author}", style="bold white underline")
                comment_content.append(f"\nProfile: {comment_profile_url}", style="dim blue")
            else:
                comment_content.append(f"{comment_author}", style="bold white")
            
            if comment_headline:
                comment_content.append(f"\n{comment_headline}", style="dim yellow")
            
            comment_content.append("\n\nComment: ", style="bold yellow")
            comment_content.append(f"{comment_text}", style="white")
            
            # Create nested panel for comment
            comment_panel = Panel(
                comment_content,
                title="[bold yellow]COMMENT[/bold yellow]",
                border_style="yellow",
                box=box.ROUNDED,
                padding=(1, 2)
            )
            comment_panels.append(comment_panel)
    
    # --- 6. CREATE MAIN CONTENT ---
    content = Text()
    content.append("User Action:  ", style="bold cyan")
    content.append(f"{header_text}\n", style="white")
    content.append("Post Author:  ", style="bold cyan")
    content.append(f"{author_name}\n", style="bold white")
    content.append("Post Author Headline: ", style="bold cyan")
    content.append(f"{author_headline}\n", style="dim white")
    content.append("Post Content: ", style="bold cyan")
    content.append(f"{post_text}", style="white")
    content.append("\nLink:         ", style="bold cyan")
    content.append(f"{post_url}", style="blue underline")
    
    # Create main panel
    if comment_panels:
        # Add separator before comments
        content.append("\n\n", style="white")
        content.append("─" * 50, style="dim")
        content.append("\n", style="white")
        
        # Create a combined renderable with main content and comment panels
        from rich.console import Group
        renderables = [content]
        renderables.extend(comment_panels)
        
        panel = Panel(
            Group(*renderables),
            title="[bold magenta]ACTIVITY FOUND[/bold magenta]",
            border_style="magenta",
            box=box.ROUNDED
        )
    else:
        panel = Panel(
            content,
            title="[bold magenta]ACTIVITY FOUND[/bold magenta]",
            border_style="magenta",
            box=box.ROUNDED
        )
    
    console.print(panel)
    console.print()  # Empty line for spacing

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Alex Hormozi                                                                                      │
│ Post Author Headline: Founder Acquisition.com, Co-Founder Skool.com. Get your free scaling roadmap👇            │
│ Post Content: You can beat 99% of people by:                                                                    │
│                                                                                                                 │
│ preparing the night before…                                                                                     │
│ showing up early…                                                                                               │
│ leaving only when the job is done exceptionally right…                                                          │
│ remembering people’s names…                                                                                     │
│ following up…                                                                                                   │
│ improving one thing each time...                                                                                │
│                                                                                                                 │
│ Do all of this consistently, every day, for one year.                                                           │
│ I promise you will get closer to your goals.                                                                    │
│                                                                                                                 │
│ - Alex ✊🏽                                                                                                       │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/alexhormozi_you-can-beat-99-of-people-by-preparing-activity-7383535744910495744- │
│ IBg5?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

urn:li:fsd_comment:(7270579058130714626,urn:li:activity:7110649546296229888)


╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein liked David Falato’s comment on this                                                │
│ Post Author:  Rob Hochstein                                                                                     │
│ Post Author Headline: Get More of Your Ideal Customers for Fast, Sustainable Revenue Growth                     │
│ Post Content: I was recently interviewed on the Garlic Marketing Show Podcast by Ian Garlic!                    │
│                                                                                                                 │
│ We discussed How LinkedIn works for B2B Outbound Sales & Cracking the Code to Waking up to Booked Sales Calls   │
│ Everyday!                                                                                                       │
│                                                                                                                 │
│ Thanks Ian for having me on and hopefully this answers some questions a lot of people ask me in regards to what │
│ actually works and some common issues that can hurt or limit your results.                                      │
│                                                                                                                 │
│ Enjoy and let me know if I can clarify anything we discussed.                                                   │
│                                                                                                                 │
│ Link to full YouTube Interview below in the comments...and I definitely  recommend subscribing to Ian's YouTube │
│ channel to hear from more successful entrepreneurs about how they are growing their businesses.                 │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/robhochstein_i-was-recently-interviewed-on-the-garlic-activity-71106495462962298 │
│ 88-DCYF?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU  │
│                                                                                                                 │
│ ──────────────────────────────────────────────────                                                              │
│                                                                                                                 │
│ ╭────────────────────────────────────────────────── COMMENT ──────────────────────────────────────────────────╮ │
│ │                                                                                                             │ │
│ │  Comment Author: David Falato                                                                               │ │
│ │  Profile: https://www.linkedin.com/in/davidfalato                                                           │ │
│ │  Empowering brands to reach their full potential                                                            │ │
│ │                                                                                                             │ │
│ │  Comment: Rob, thanks for sharing!  How are you?                                                            │ │
│ │                                                                                                             │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

urn:li:fsd_comment:(7316475933408874497,urn:li:activity:7110649546296229888)


╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein liked Lauren Apple 🍏’s comment on this                                             │
│ Post Author:  Rob Hochstein                                                                                     │
│ Post Author Headline: Get More of Your Ideal Customers for Fast, Sustainable Revenue Growth                     │
│ Post Content: I was recently interviewed on the Garlic Marketing Show Podcast by Ian Garlic!                    │
│                                                                                                                 │
│ We discussed How LinkedIn works for B2B Outbound Sales & Cracking the Code to Waking up to Booked Sales Calls   │
│ Everyday!                                                                                                       │
│                                                                                                                 │
│ Thanks Ian for having me on and hopefully this answers some questions a lot of people ask me in regards to what │
│ actually works and some common issues that can hurt or limit your results.                                      │
│                                                                                                                 │
│ Enjoy and let me know if I can clarify anything we discussed.                                                   │
│                                                                                                                 │
│ Link to full YouTube Interview below in the comments...and I definitely  recommend subscribing to Ian's YouTube │
│ channel to hear from more successful entrepreneurs about how they are growing their businesses.                 │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/robhochstein_i-was-recently-interviewed-on-the-garlic-activity-71106495462962298 │
│ 88-DCYF?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU  │
│                                                                                                                 │
│ ──────────────────────────────────────────────────                                                              │
│                                                                                                                 │
│ ╭────────────────────────────────────────────────── COMMENT ──────────────────────────────────────────────────╮ │
│ │                                                                                                             │ │
│ │  Comment Author: Lauren Apple 🍏                                                                            │ │
│ │  Profile: https://www.linkedin.com/in/lauren-apple                                                          │ │
│ │  I help CEO Parents scale your business AND your margin | 5x Mom & CEO | Scaling Strategist for CEO         │ │
│ │  Parents                                                                                                    │ │
│ │                                                                                                             │ │
│ │  Comment: Sounds like an insightful conversation! Looking forward to checking it out. 😊                    │ │
│ │                                                                                                             │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Lindsay Christensen                                                                               │
│ Post Author Headline: Premier IV Hydration Therapy | Anti-Aging | Weight loss | Hangover Recovery | Athletic    │
│ Performance | Immunity Armour | Fatigue Fighter | Jet Lag Recovery                                              │
│ Post Content: I’m happy to share that I’m opening my first business: Prime IV Hydration & Wellness (Geneva,     │
│ IL). Super grateful to Matt for always having my back and giving me the flexibility to find my next adventure❤️  │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/linz-christensen_im-happy-to-share-that-im-opening-my-first-ugcPost-723546391258 │
│ 0595712-YdBD?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfq │
│ NIfU                                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Justin Welsh                                                                                      │
│ Post Author Headline: The $10M Solopreneur | Helping 100,000+ experts turn their expertise into income.         │
│ Post Content: There are 8 simple daily activities that changed my life.                                         │
│                                                                                                                 │
│ (and may change yours)                                                                                          │
│                                                                                                                 │
│ 1. A morning workout                                                                                            │
│ 2. A big, healthy lunch                                                                                         │
│ 3. A small, healthy dinner                                                                                      │
│ 4. 3 hours in the 'genius' zone                                                                                 │
│ 5. 30 minutes of reading                                                                                        │
│ 6. Time with my wife                                                                                            │
│ 7. Time outside                                                                                                 │
│ 8. Time alone                                                                                                   │
│                                                                                                                 │
│ My best days as an entrepreneur have all 8.                                                                     │
│                                                                                                                 │
│ I call this 'life-work integration'.                                                                            │
│                                                                                                                 │
│ Here's how you build it: https://lnkd.in/ea7MU6-8                                                               │
│                                                                                                                 │
│ What do your best days look like?                                                                               │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/justinwelsh_there-are-8-simple-daily-activities-that-activity-722872917276273459 │
│ 2-lyDm?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Matt Orlins                                                                                       │
│ Post Author Headline: Government Relations | Cross-Functional Leadership | Storytelling | Attorney              │
│ Post Content: This week, the U.S. Senate approved versions of the Kids Online Safety Act and the Children’s     │
│ Online Privacy Protection Act, which would represent some of the most significant legislation in decades aimed  │
│ at addressing how platforms can interact with children. Passage in the House remains at best an open question.  │
│ Some members are already expressing serious reservations.                                                       │
│                                                                                                                 │
│ As someone who works in early childhood education policy and as a parent, I am interested to see what approach, │
│ if any, states and the federal government coalesce around to ensure that kids do not experience so many adverse │
│ consequences from spending time online. A couple of weeks ago, Kernard D. Jones and I hosted a roundtable on    │
│ screen usage and mental health focused on early learners at the Education Commission of the States National     │
│ Forum on Education Policy. One takeaway for us is that there’s a ton of interest and energy to do something,    │
│ but that we’re still looking for consensus on what that is.                                                     │
│                                                                                                                 │
│ This week’s legislative wrangling is a reminder, though, that as tech platforms and parents, we don’t have to   │
│ wait. There are things that we can and should do to try to help ensure that any experiences kids do have online │
│ are positive and age appropriate. One of the reasons that I’m proud to work at Waterford.org is that we center  │
│ children in the design of our programs.  Our programs are time limited, highly interactive, age appropriate,    │
│ rooted in the science of learning, and offered free of advertising. Technology can continue to be an essential  │
│ tool for helping children learn, but we need to be proactive to ensure that it is.                              │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/mattorlins_best-practices-for-screen-time-in-early-childhood-ugcPost-72251309146 │
│ 26842625-1Kdt?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWf │
│ qNIfU                                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Justin Welsh                                                                                      │
│ Post Author Headline: The $10M Solopreneur | Helping 100,000+ experts turn their expertise into income.         │
│ Post Content: Stop waiting for your "big break."                                                                │
│                                                                                                                 │
│ Go create it yourself.                                                                                          │
│                                                                                                                 │
│ Most people I know:                                                                                             │
│                                                                                                                 │
│ • Chase trends                                                                                                  │
│ • Wait for opportunities                                                                                        │
│ • Hope for luck                                                                                                 │
│                                                                                                                 │
│ The result?                                                                                                     │
│                                                                                                                 │
│ A future decided by others.                                                                                     │
│                                                                                                                 │
│ Here's a better plan:                                                                                           │
│                                                                                                                 │
│ • Set the trends                                                                                                │
│ • Create opportunities                                                                                          │
│ • Make your own damn luck                                                                                       │
│                                                                                                                 │
│ Want to be successful in 5 years?                                                                               │
│ Start building that success today.                                                                              │
│                                                                                                                 │
│ It's not about predicting the market.                                                                           │
│ It's about *creating* the market.                                                                               │
│                                                                                                                 │
│ And easiest place is right here on LinkedIn.                                                                    │
│                                                                                                                 │
│ Here's how 25,000+ people have done the same: https://lnkd.in/egPyfFmy                                          │
│                                                                                                                 │
│ What future will you start creating today?            

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Leila Hormozi                                                                                     │
│ Post Author Headline: Founder and CEO of Acquisition.com                                                        │
│ Post Content: Just like a fish grows to the limit of its tank.                                                  │
│ A business grows to the limit of one thing.                                                                     │
│                                                                                                                 │
│ The capability of the person leading the company.                                                               │
│ A business grows to the limit of its leader's capability.                                                       │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/leilahormozi_just-like-a-fish-grows-to-the-limit-of-its-activity-722735037931364 │
│ 3521-At34?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIf │
│ U                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [32]:
from rich.console import Console
from rich.panel import Panel
from rich.text import Text
from rich import box
from rich.console import Group

# 1. Create lookup index for speed
index = {item.get('entityUrn'): item for item in json_data['included'] if 'entityUrn' in item}

# 2. Extract activities
activities = json_data.get('data', {}).get('data', {}).get('feedDashProfileUpdatesByMemberReactions', {}).get('*elements', [])

# Initialize rich console
console = Console()

def create_comment_panel(comment_obj, index, is_reply=False):
    """Helper function to create a comment panel with nested replies"""
    # Extract comment data
    comment_text = comment_obj.get('commentary', {}).get('text', {})
    commenter = comment_obj.get('commenter', {})
    comment_author = commenter.get('title', {}).get('text', "Unknown")
    comment_profile_url = commenter.get('navigationUrl', "")
    comment_headline = commenter.get('subtitle', "")
    
    # Create comment content with author info
    comment_content = Text()
    comment_content.append("Comment Author: ", style="bold yellow" if not is_reply else "bold green")
    if comment_profile_url:
        comment_content.append(f"{comment_author}", style="bold white underline")
        comment_content.append(f"\nProfile: {comment_profile_url}", style="dim blue")
    else:
        comment_content.append(f"{comment_author}", style="bold white")
    
    if comment_headline:
        comment_content.append(f"\n{comment_headline}", style="dim yellow" if not is_reply else "dim green")
    
    comment_content.append("\n\nComment: ", style="bold yellow" if not is_reply else "bold green")
    comment_content.append(f"{comment_text}", style="white")
    
    # Get replies for this comment
    reply_panels = []
    social_detail_urn = comment_obj.get('*socialDetail')
    if social_detail_urn:
        social_detail = index.get(social_detail_urn)
        if social_detail and social_detail.get('comments'):
            reply_urns = social_detail.get('comments', {}).get('*elements', [])
            for reply_urn in reply_urns:
                reply_obj = index.get(reply_urn)
                if reply_obj:
                    reply_panel = create_comment_panel(reply_obj, index, is_reply=True)
                    reply_panels.append(reply_panel)
    
    # Create panel for this comment
    border_style = "yellow" if not is_reply else "green"
    title = "[bold yellow]COMMENT[/bold yellow]" if not is_reply else "[bold green]REPLY[/bold green]"
    
    if reply_panels:
        # If there are replies, nest them inside
        comment_content.append("\n\n", style="white")
        comment_content.append("─" * 40, style="dim")
        comment_content.append("\n", style="white")
        
        # Create a group with comment content and reply panels
        renderables = [comment_content]
        renderables.extend(reply_panels)
        
        comment_panel = Panel(
            Group(*renderables),
            title=title,
            border_style=border_style,
            box=box.ROUNDED,
            padding=(1, 2)
        )
    else:
        comment_panel = Panel(
            comment_content,
            title=title,
            border_style=border_style,
            box=box.ROUNDED,
            padding=(1, 2)
        )
    
    return comment_panel

for urn in activities:
    item = index.get(urn)
    if not item or item.get('$type') != 'com.linkedin.voyager.dash.feed.Update':
        continue

    # --- 1. THE ACTION & REACTION TYPE ---
    header_text = item.get('header', {}).get('text', {}).get('text', "")

    # --- 2. THE ORIGINAL POST TEXT ---
    # In a reaction update, the original post text is often in 'commentary'
    # or inside 'resharedUpdate' if they reacted to a share.
    post_text = "No text content"
    if item.get('commentary'):
        post_text = item['commentary'].get('text', {}).get('text', "")
    
    # --- 3. THE ORIGINAL AUTHOR ---
    # We look for the 'actor' URN and find their name in our index
    author_name = "Unknown Author"
    author_headline = ""
    actor = item.get('actor')
    if actor:
        author_name = actor.get("name", {}).get("text", "Unknown Author")
        author_headline = actor.get("description", {}).get("text", "")

    # --- 4. THE URL ---
    post_url = item.get('socialContent', {}).get('shareUrl') or item.get('metadata', {}).get('shareUrl')

    # --- 5. EXTRACT COMMENTS WITH REPLIES ---
    highlightedComments = item.get('*highlightedComments', [])
    comment_panels = []
    
    if highlightedComments:
        for comment_urn in highlightedComments:
            comment_obj = index.get(comment_urn)
            if not comment_obj:
                continue
            
            # Create comment panel (which will recursively include replies)
            comment_panel = create_comment_panel(comment_obj, index, is_reply=False)
            comment_panels.append(comment_panel)
    
    # --- 6. CREATE MAIN CONTENT ---
    content = Text()
    content.append("User Action:  ", style="bold cyan")
    content.append(f"{header_text}\n", style="white")
    content.append("Post Author:  ", style="bold cyan")
    content.append(f"{author_name}\n", style="bold white")
    content.append("Post Author Headline: ", style="bold cyan")
    content.append(f"{author_headline}\n", style="dim white")
    content.append("Post Content: ", style="bold cyan")
    content.append(f"{post_text}", style="white")
    content.append("\nLink:         ", style="bold cyan")
    content.append(f"{post_url}", style="blue underline")
    
    # Create main panel
    if comment_panels:
        # Add separator before comments
        content.append("\n\n", style="white")
        content.append("─" * 50, style="dim")
        content.append("\n", style="white")
        
        # Create a combined renderable with main content and comment panels
        renderables = [content]
        renderables.extend(comment_panels)
        
        panel = Panel(
            Group(*renderables),
            title="[bold magenta]ACTIVITY FOUND[/bold magenta]",
            border_style="magenta",
            box=box.ROUNDED
        )
    else:
        panel = Panel(
            content,
            title="[bold magenta]ACTIVITY FOUND[/bold magenta]",
            border_style="magenta",
            box=box.ROUNDED
        )
    
    console.print(panel)
    console.print()  # Empty line for spacing

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Alex Hormozi                                                                                      │
│ Post Author Headline: Founder Acquisition.com, Co-Founder Skool.com. Get your free scaling roadmap👇            │
│ Post Content: You can beat 99% of people by:                                                                    │
│                                                                                                                 │
│ preparing the night before…                                                                                     │
│ showing up early…                                                                                               │
│ leaving only when the job is done exceptionally right…                                                          │
│ remembering people’s names…                                                                                     │
│ following up…                                                                                                   │
│ improving one thing each time...                                                                                │
│                                                                                                                 │
│ Do all of this consistently, every day, for one year.                                                           │
│ I promise you will get closer to your goals.                                                                    │
│                                                                                                                 │
│ - Alex ✊🏽                                                                                                       │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/alexhormozi_you-can-beat-99-of-people-by-preparing-activity-7383535744910495744- │
│ IBg5?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein liked David Falato’s comment on this                                                │
│ Post Author:  Rob Hochstein                                                                                     │
│ Post Author Headline: Get More of Your Ideal Customers for Fast, Sustainable Revenue Growth                     │
│ Post Content: I was recently interviewed on the Garlic Marketing Show Podcast by Ian Garlic!                    │
│                                                                                                                 │
│ We discussed How LinkedIn works for B2B Outbound Sales & Cracking the Code to Waking up to Booked Sales Calls   │
│ Everyday!                                                                                                       │
│                                                                                                                 │
│ Thanks Ian for having me on and hopefully this answers some questions a lot of people ask me in regards to what │
│ actually works and some common issues that can hurt or limit your results.                                      │
│                                                                                                                 │
│ Enjoy and let me know if I can clarify anything we discussed.                                                   │
│                                                                                                                 │
│ Link to full YouTube Interview below in the comments...and I definitely  recommend subscribing to Ian's YouTube │
│ channel to hear from more successful entrepreneurs about how they are growing their businesses.                 │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/robhochstein_i-was-recently-interviewed-on-the-garlic-activity-71106495462962298 │
│ 88-DCYF?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU  │
│                                                                                                                 │
│ ──────────────────────────────────────────────────                                                              │
│                                                                                                                 │
│ ╭────────────────────────────────────────────────── COMMENT ──────────────────────────────────────────────────╮ │
│ │                                                                                                             │ │
│ │  Comment Author: David Falato                                                                               │ │
│ │  Profile: https://www.linkedin.com/in/davidfalato                                                           │ │
│ │  Empowering brands to reach their full potential                                                            │ │
│ │                                                                                                             │ │
│ │  Comment: Rob, thanks for sharing!  How are you?                                                            │ │
│ │                                                                                                             │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein liked Lauren Apple 🍏’s comment on this                                             │
│ Post Author:  Rob Hochstein                                                                                     │
│ Post Author Headline: Get More of Your Ideal Customers for Fast, Sustainable Revenue Growth                     │
│ Post Content: I was recently interviewed on the Garlic Marketing Show Podcast by Ian Garlic!                    │
│                                                                                                                 │
│ We discussed How LinkedIn works for B2B Outbound Sales & Cracking the Code to Waking up to Booked Sales Calls   │
│ Everyday!                                                                                                       │
│                                                                                                                 │
│ Thanks Ian for having me on and hopefully this answers some questions a lot of people ask me in regards to what │
│ actually works and some common issues that can hurt or limit your results.                                      │
│                                                                                                                 │
│ Enjoy and let me know if I can clarify anything we discussed.                                                   │
│                                                                                                                 │
│ Link to full YouTube Interview below in the comments...and I definitely  recommend subscribing to Ian's YouTube │
│ channel to hear from more successful entrepreneurs about how they are growing their businesses.                 │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/robhochstein_i-was-recently-interviewed-on-the-garlic-activity-71106495462962298 │
│ 88-DCYF?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU  │
│                                                                                                                 │
│ ──────────────────────────────────────────────────                                                              │
│                                                                                                                 │
│ ╭────────────────────────────────────────────────── COMMENT ──────────────────────────────────────────────────╮ │
│ │                                                                                                             │ │
│ │  Comment Author: Lauren Apple 🍏                                                                            │ │
│ │  Profile: https://www.linkedin.com/in/lauren-apple                                                          │ │
│ │  I help CEO Parents scale your business AND your margin | 5x Mom & CEO | Scaling Strategist for CEO         │ │
│ │  Parents                                                                                                    │ │
│ │                                                                                                             │ │
│ │  Comment: Sounds like an insightful conversation! Looking forward to checking it out. 😊                    │ │
│ │                                                                                                             │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Marcus Hayes                                                                                      │
│ Post Author Headline: Chief Estimator & Owner                                                                   │
│ Post Content: Big thanks to my team for their efforts on this project, and to Layton Construction for the       │
│ opportunity participate.                                                                                        │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/marcuskeithhayes_carmichaelcollege-vandyhre-commercialpainters-ugcPost-737276694 │
│ 0710428672-rc-T?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGG │
│ WfqNIfU                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein liked Daniel Lewkovitz’s comment on this                                            │
│ Post Author:  John Menadue                                                                                      │
│ Post Author Headline: Editor-In-Chief at Pearls and Irritations Ptd Ltd                                         │
│ Post Content: Antisemitism did not spring up here as suddenly and as localised as a field of mushrooms. It is,  │
│ above all, a by-product of Israel’s endless onslaught on the people of Gaza which one and all can watch as a    │
│ daily horror show.                                                                                              │
│ By: Henry Reynolds                                                                                              │
│ https://lnkd.in/gPJSdxug                                                                                        │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/john-menadue_conflation-and-controversy-over-antisemitism-activity-7335425171744 │
│ 272384-wCHs?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqN │
│ IfU                                                                                                             │
│                                                                                                                 │
│ ──────────────────────────────────────────────────                                                              │
│                                                                                                                 │
│ ╭────────────────────────────────────────────────── COMMENT ──────────────────────────────────────────────────╮ │
│ │                                                                                                             │ │
│ │  Comment Author: Danielle Farrow-Pryke                                                                      │ │
│ │  Profile: https://www.linkedin.com/in/danielle-farrow-pryke-3b597027                                        │ │
│ │  BCom (UTS) | IAP2-Certified | Policy Campaigner | AFTRS V/O+Professional Singer, Actor & Writer |          │ │
│ │  Educator | Legislative Reform | DV | YouTube: Australian Feminist | PUBLIC SNAPCHAT: aussiefeminist        │ │
│ │                                                                                                             │ │
│ │  Comment: By definition, a genocide is where people are targeted due to their race or ethnicity. We’ll set  │ │
│ │  aside the fact that the Jews didn’t even register a blank compared to Stalin‘s 22 million. That 22         │ │
│ │  million happened to be anybody who opposed him. Was that a genocide? I guess you’re too young, or stupid,  │ │
│ │  to remember the Khmer Rouge? Modern day Myanmar? Kosovo’s muslims 3 decades ago? But it’s not about        │ │
│ │  numbers is it. It’s about your outrageous privilege. Did you have a theory while we’re at it on why        │ │
│ │  Europe tried to exterminate your people? I would be fascinated to hear your response. Genuine question.    │ │
│ │                                                                                                             │ │
│ │  ────────────────────────────────────────                                                                   │ │
│ │                                                                                                             │ │
│ │  ╭──────────────────────────────────────────────── REPLY ────────────────────────────────────────────────╮  │ │
│ │  │                                                                                                       │  │ │
│ │  │  Comment Author: Daniel Lewkovitz                

...

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Lindsay Christensen                                                                               │
│ Post Author Headline: Premier IV Hydration Therapy | Anti-Aging | Weight loss | Hangover Recovery | Athletic    │
│ Performance | Immunity Armour | Fatigue Fighter | Jet Lag Recovery                                              │
│ Post Content: I’m happy to share that I’m opening my first business: Prime IV Hydration & Wellness (Geneva,     │
│ IL). Super grateful to Matt for always having my back and giving me the flexibility to find my next adventure❤️  │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/linz-christensen_im-happy-to-share-that-im-opening-my-first-ugcPost-723546391258 │
│ 0595712-YdBD?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfq │
│ NIfU                                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Justin Welsh                                                                                      │
│ Post Author Headline: The $10M Solopreneur | Helping 100,000+ experts turn their expertise into income.         │
│ Post Content: There are 8 simple daily activities that changed my life.                                         │
│                                                                                                                 │
│ (and may change yours)                                                                                          │
│                                                                                                                 │
│ 1. A morning workout                                                                                            │
│ 2. A big, healthy lunch                                                                                         │
│ 3. A small, healthy dinner                                                                                      │
│ 4. 3 hours in the 'genius' zone                                                                                 │
│ 5. 30 minutes of reading                                                                                        │
│ 6. Time with my wife                                                                                            │
│ 7. Time outside                                                                                                 │
│ 8. Time alone                                                                                                   │
│                                                                                                                 │
│ My best days as an entrepreneur have all 8.                                                                     │
│                                                                                                                 │
│ I call this 'life-work integration'.                                                                            │
│                                                                                                                 │
│ Here's how you build it: https://lnkd.in/ea7MU6-8                                                               │
│                                                                                                                 │
│ What do your best days look like?                                                                               │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/justinwelsh_there-are-8-simple-daily-activities-that-activity-722872917276273459 │
│ 2-lyDm?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIfU   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Matt Orlins                                                                                       │
│ Post Author Headline: Government Relations | Cross-Functional Leadership | Storytelling | Attorney              │
│ Post Content: This week, the U.S. Senate approved versions of the Kids Online Safety Act and the Children’s     │
│ Online Privacy Protection Act, which would represent some of the most significant legislation in decades aimed  │
│ at addressing how platforms can interact with children. Passage in the House remains at best an open question.  │
│ Some members are already expressing serious reservations.                                                       │
│                                                                                                                 │
│ As someone who works in early childhood education policy and as a parent, I am interested to see what approach, │
│ if any, states and the federal government coalesce around to ensure that kids do not experience so many adverse │
│ consequences from spending time online. A couple of weeks ago, Kernard D. Jones and I hosted a roundtable on    │
│ screen usage and mental health focused on early learners at the Education Commission of the States National     │
│ Forum on Education Policy. One takeaway for us is that there’s a ton of interest and energy to do something,    │
│ but that we’re still looking for consensus on what that is.                                                     │
│                                                                                                                 │
│ This week’s legislative wrangling is a reminder, though, that as tech platforms and parents, we don’t have to   │
│ wait. There are things that we can and should do to try to help ensure that any experiences kids do have online │
│ are positive and age appropriate. One of the reasons that I’m proud to work at Waterford.org is that we center  │
│ children in the design of our programs.  Our programs are time limited, highly interactive, age appropriate,    │
│ rooted in the science of learning, and offered free of advertising. Technology can continue to be an essential  │
│ tool for helping children learn, but we need to be proactive to ensure that it is.                              │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/mattorlins_best-practices-for-screen-time-in-early-childhood-ugcPost-72251309146 │
│ 26842625-1Kdt?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWf │
│ qNIfU                                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Justin Welsh                                                                                      │
│ Post Author Headline: The $10M Solopreneur | Helping 100,000+ experts turn their expertise into income.         │
│ Post Content: Stop waiting for your "big break."                                                                │
│                                                                                                                 │
│ Go create it yourself.                                                                                          │
│                                                                                                                 │
│ Most people I know:                                                                                             │
│                                                                                                                 │
│ • Chase trends                                                                                                  │
│ • Wait for opportunities                                                                                        │
│ • Hope for luck                                                                                                 │
│                                                                                                                 │
│ The result?                                                                                                     │
│                                                                                                                 │
│ A future decided by others.                                                                                     │
│                                                                                                                 │
│ Here's a better plan:                                                                                           │
│                                                                                                                 │
│ • Set the trends                                                                                                │
│ • Create opportunities                                                                                          │
│ • Make your own damn luck                                                                                       │
│                                                                                                                 │
│ Want to be successful in 5 years?                                                                               │
│ Start building that success today.                                                                              │
│                                                                                                                 │
│ It's not about predicting the market.                                                                           │
│ It's about *creating* the market.                                                                               │
│                                                                                                                 │
│ And easiest place is right here on LinkedIn.                                                                    │
│                                                                                                                 │
│ Here's how 25,000+ people have done the same: https://lnkd.in/egPyfFmy                                          │
│                                                                                                                 │
│ What future will you start creating today?            

╭──────────────────────────────────────────────── ACTIVITY FOUND ─────────────────────────────────────────────────╮
│ User Action:  Rob Hochstein likes this                                                                          │
│ Post Author:  Leila Hormozi                                                                                     │
│ Post Author Headline: Founder and CEO of Acquisition.com                                                        │
│ Post Content: Just like a fish grows to the limit of its tank.                                                  │
│ A business grows to the limit of one thing.                                                                     │
│                                                                                                                 │
│ The capability of the person leading the company.                                                               │
│ A business grows to the limit of its leader's capability.                                                       │
│ Link:                                                                                                           │
│ https://www.linkedin.com/posts/leilahormozi_just-like-a-fish-grows-to-the-limit-of-its-activity-722735037931364 │
│ 3521-At34?utm_source=social_share_send&utm_medium=member_desktop_web&rcm=ACoAAEoOn8sBnFFiwG6I-uBI2sHeQCGGWfqNIf │
│ U                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯